# Concat-зв'язок (конкатенація) з нуля + backprop руками

Друга частина в стилі [Karpathy — «Becoming a Backprop Ninja»](https://www.youtube.com/watch?v=q8SA3rM6ckI),
лише numpy. Тепер skip-зв'язок як у **U-Net**: ознаки з раннього шару не додаються, а **склеюються**
(concat) з глибокими: `c = concat([a, v])`.

У U-Net склеювання йде по осі **каналів**: у кожному пікселі вектор ознак encoder і вектор ознак decoder
стають одним довшим вектором. Тут «пікселів» немає — кожен рядок `a` і `v` і є таким вектором ознак,
а повнозв'язний шар після concat — це аналог згортки 1×1.

Задача-іграшка та сама: точки на площині, клас 1 — всередині кола.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

SEED = 42
rng = np.random.default_rng(SEED)

N = 200
x = rng.uniform(-1, 1, size=(N, 2))
y = ((x ** 2).sum(axis=1, keepdims=True) < 0.5).astype(np.float64)

plt.figure(figsize=(4, 4))
plt.scatter(x[:, 0], x[:, 1], c=y[:, 0], cmap="coolwarm", s=15)
plt.title("1 — всередині кола, 0 — зовні")
plt.gca().set_aspect("equal")
plt.show()
print("x:", x.shape, "y:", y.shape, "частка класу 1:", y.mean().round(2))

## Ланцюгове правило (chain rule) і локальні градієнти

Позначення як у Karpathy: `dX` — це **∂loss/∂X**, градієнт loss по змінній `X`, тієї самої форми, що й `X`.

Backward іде від loss назад. Для кожної операції `out = f(in)` знаємо `dout` (прийшов «згори») і рахуємо
`din = dout · ∂out/∂in` — це **ланцюгове правило**. Тобто кожній операції досить знати лише свою
**локальну похідну**:

| Операція (forward) | Backward | Інтуїція |
|---|---|---|
| `out = X @ W` | `dX = dout @ W.T`, `dW = X.T @ dout` | форми підказують порядок множення |
| `out = X + b` (b — рядок, broadcasting) | `dX = dout`, `db = dout.sum(axis=0)` | b додано до кожного рядка → градієнти рядків сумуються |
| `out = relu(h)` | `dh = dout * (h > 0)` | пропускає градієнт, де h > 0, і «глушить» де h ≤ 0 |
| `out = a + v` | `da = dout`, `dv = dout` | **додавання копіює градієнт** в обидві гілки |
| `out = concat([a, v])` | `da = dout[:, :H]`, `dv = dout[:, H:]` | **конкатенація розрізає градієнт** на шматки |
| loss = BCE(sigmoid(z), y), середнє по N | `dz = (sigmoid(z) - y) / N` | «наскільки p відрізняється від правди» |

**Правило розгалуження:** якщо змінна використовується **двічі** (тут `a` — і в skip, і в гілці),
її градієнт — **сума** градієнтів з обох місць. Тому в коді `da = ...`, а потім `da += ...`.

## Мережа

```
x ─► h = x @ W1 + b1 ─► a = relu(h) ─┬─────────────────────────────────────┐  skip
                                     └─► u = a @ W2 + b2 ─► v = relu(u) ───┴─► c = concat([a, v]) ─► z = c @ W3 + b3 ─► loss
```

На відміну від додавання, `a` і `v` **не мусять** мати однакову ширину: тут `a` — 16 ознак, `v` — 8,
`c` — 24. Ціна — більший наступний шар: `W3` має форму `(16 + 8, 1)`.

In [ ]:
H = 16
H2 = 8


def init_params():
    rng = np.random.default_rng(SEED)
    return {
        "W1": rng.normal(size=(2, H)) * np.sqrt(2 / 2),
        "b1": rng.normal(size=H) * 0.1,
        "W2": rng.normal(size=(H, H2)) * np.sqrt(2 / H),
        "b2": rng.normal(size=H2) * 0.1,
        "W3": rng.normal(size=(H + H2, 1)) * np.sqrt(1 / (H + H2)),
        "b3": rng.normal(size=1) * 0.1,
    }


def forward(params, x, y):
    h = x @ params["W1"] + params["b1"]
    a = np.maximum(h, 0)
    u = a @ params["W2"] + params["b2"]
    v = np.maximum(u, 0)
    c = np.concatenate([a, v], axis=1)
    z = c @ params["W3"] + params["b3"]
    # Стабільна форма BCE з логітів: без exp від великих чисел.
    loss = np.mean(np.maximum(z, 0) - z * y + np.log1p(np.exp(-np.abs(z))))
    return loss, {"x": x, "h": h, "a": a, "u": u, "v": v, "c": c, "z": z}


params = init_params()
loss, cache = forward(params, x, y)
for name, value in cache.items():
    print(f"{name}: {value.shape}")
print("loss:", round(loss, 4))

## Backward — рядок за рядком

1. `dz = (sigmoid(z) - y) / N`.
2. `z = c @ W3 + b3` → `dW3`, `db3`, `dc` форми `(N, 24)`.
3. **`c = concat([a, v])` → `da = dc[:, :16]`, `dv = dc[:, 16:]`.** Конкатенація в backward —
   це **розрізання**: кожна частина отримує свій шматок градієнта, рівно тієї форми, що й у forward.
4. `v = relu(u)` → `du`; `u = a @ W2 + b2` → `dW2`, `db2` і другий внесок у `da`.
5. `da` = (зі skip) + (з гілки) — те саме правило розгалуження, що й у residual.
6. `a = relu(h)` → `dh`; `h = x @ W1 + b1` → `dW1`, `db1`.

Порівняй з residual: там `da` зі skip **дорівнює** `dr` (копія), тут — **свій шматок** `dc`,
який залежить від «своїх» рядків `W3`.

In [ ]:
def backward(params, cache, y):
    x, h, a, u, v, c, z = (cache[k] for k in ["x", "h", "a", "u", "v", "c", "z"])
    n = len(y)

    dz = (1 / (1 + np.exp(-z)) - y) / n
    dW3 = c.T @ dz
    db3 = dz.sum(axis=0)
    dc = dz @ params["W3"].T

    da = dc[:, :H].copy()
    dv = dc[:, H:]

    du = dv * (u > 0)
    dW2 = a.T @ du
    db2 = du.sum(axis=0)
    da += du @ params["W2"].T

    dh = da * (h > 0)
    dW1 = x.T @ dh
    db1 = dh.sum(axis=0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2, "W3": dW3, "b3": db3}


grads = backward(params, cache, y)
for name, grad in grads.items():
    print(f"d{name}: {grad.shape} (як у {name}: {params[name].shape})")

## Перевірка: числовий градієнт (gradient check)

Karpathy перевіряє свої градієнти через PyTorch autograd. Ми обійдемось без бібліотек —
**числовим градієнтом** за означенням похідної (центральна різниця):

∂loss/∂w ≈ (loss(w + ε) − loss(w − ε)) / 2ε, ε = 1e-6

Для кожного числа в кожній матриці ваг трохи зсуваємо його вгору і вниз і дивимось, як змінився loss.
Це повільно (2 forward на кожну вагу), але тривіально правильно — ідеальний «еталон» для перевірки.
Усе в `float64`: у `float32` різниця loss(w+ε) − loss(w−ε) губиться в округленні.

Відносна похибка ~1e-7 і менше — градієнт правильний; ~1e-2 і більше — помилка в backward.

In [ ]:
def numerical_grad(params, name, eps=1e-6):
    param = params[name]
    grad = np.zeros_like(param)
    for idx in np.ndindex(param.shape):
        old = param[idx]
        param[idx] = old + eps
        loss_plus, _ = forward(params, x, y)
        param[idx] = old - eps
        loss_minus, _ = forward(params, x, y)
        param[idx] = old
        grad[idx] = (loss_plus - loss_minus) / (2 * eps)
    return grad


def cmp(name, manual, numeric):
    max_diff = np.abs(manual - numeric).max()
    rel_error = max_diff / (np.abs(manual).max() + np.abs(numeric).max() + 1e-12)
    status = "OK" if rel_error < 1e-6 else "ПОМИЛКА"
    print(f"{name:3s} | max різниця {max_diff:.2e} | відносна похибка {rel_error:.2e} | {status}")


for name in params:
    cmp(name, grads[name], numerical_grad(params, name))

**Експеримент для самоперевірки:** заміни `dc[:, :H]` на `dc[:, H:]` і навпаки (переплутати, де який
шматок — реальна помилка при concat у U-Net) і запусти перевірку знову. Що зламається?

## Навчання: градієнтний спуск руками

Тепер, коли градієнти правильні, навчання — це просто цикл: forward → backward → `param -= lr * grad`
(звичайний градієнтний спуск, без оптимізаторів з бібліотек). Потім малюємо межу рішень (decision boundary):
де мережа каже «всередині кола».

In [ ]:
LR = 0.5
STEPS = 3000

for step in range(STEPS + 1):
    loss, cache = forward(params, x, y)
    grads = backward(params, cache, y)
    for name in params:
        params[name] -= LR * grads[name]
    if step % 500 == 0:
        accuracy = ((cache["z"] > 0) == (y > 0.5)).mean()
        print(f"крок {step:4d}: loss {loss:.4f}, точність {accuracy:.1%}")

In [ ]:
grid_1d = np.linspace(-1, 1, 200)
grid = np.array(np.meshgrid(grid_1d, grid_1d)).reshape(2, -1).T
_, grid_cache = forward(params, grid, np.zeros((len(grid), 1)))
inside = (grid_cache["z"] > 0).reshape(200, 200)

plt.figure(figsize=(4, 4))
plt.contourf(grid_1d, grid_1d, inside, alpha=0.3, cmap="coolwarm")
plt.scatter(x[:, 0], x[:, 1], c=y[:, 0], cmap="coolwarm", s=15)
plt.gca().set_aspect("equal")
plt.title("межа рішень після навчання")
plt.show()

## Concat + шар = додавання з окремими вагами

Розріжемо `W3` на верхню частину (рядки для `a`) і нижню (для `v`):

`concat([a, v]) @ W3 = a @ W3_top + v @ W3_bottom`

Тобто concat з наступним шаром — це **додавання, але кожна частина спершу множиться на свої ваги**.
Residual `(a + v) @ W` — окремий випадок, коли `W3_top = W3_bottom = W` (і ширини однакові).
Тому concat **гнучкіший**: мережа сама вирішує, скільки брати з ранніх ознак (деталі, «де»)
і скільки з глибоких («що»). Ціна — більше ваг і пам'яті (канали подвоюються).

Перевіримо числами:

In [ ]:
a, v = cache["a"], cache["v"]
W3 = params["W3"]
via_concat = np.concatenate([a, v], axis=1) @ W3
via_sum = a @ W3[:H] + v @ W3[H:]
print("макс. різниця:", np.abs(via_concat - via_sum).max())

## Додавання vs конкатенація — підсумок

| | Додавання (ResNet) | Конкатенація (U-Net, DenseNet) |
|---|---|---|
| Forward | `a + v` | `concat([a, v])` |
| Форми | мають збігатись | можуть різнитись (по осі склеювання) |
| Backward | **копія** `dout` в обидві гілки | **розрізання** `dout` на шматки |
| Наступний шар | бачить суміш — не знає, що звідки | бачить обидва окремо, ваги для кожного свої |
| Ціна | безкоштовно | більше каналів → більше ваг і пам'яті |
| Навіщо | градієнт іде крізь глибину без множень | повернути деталі з ранніх шарів |

## На співбесіді

- **«Як градієнт проходить через concat?»** Розрізається на шматки за тими самими розмірами,
  що були у forward; кожен вхід отримує свій шматок.
- **«Чому в U-Net concat, а не додавання?»** Ранні ознаки (краї, точні межі) і глибокі (семантика)
  — різні за змістом; concat дає наступному шару бачити їх окремо й зважувати по-своєму.
  Додавання змішало б їх в одне число на канал. Ціна — більше каналів і пам'яті.
- **«Чи можна виразити додавання через concat?»** Так: concat + лінійний шар =
  `a @ W_top + v @ W_bottom`; додавання — випадок `W_top = W_bottom`.